# 滚动吸附与滚动驱动动画

学习目标：能配置可操作的滚动吸附，并把装饰动画关联到滚动或视图进度，同时响应减少动画偏好。

前置知识：滚动容器、Flexbox、定位、关键帧动画、媒体查询与键盘焦点。

适用范围：CSS Scroll Snap Level 1 与演进中的 Scroll-driven Animations Level 1。滚动时间线、视图时间线和动画范围需分别检查目标浏览器支持；本章不使用 JavaScript。

工作目录：`content/Web与应用开发/css/`（以下命令从项目根目录切换到这里执行）。

环境准备：[环境配置与运行](README.md)。

配套脚本：位于 scripts/26-scroll-snap-and-driven-animations/。

1. [index.html](scripts/26-scroll-snap-and-driven-animations/index.html)：卡片吸附、长内容和滚动链。
2. [animations.html](scripts/26-scroll-snap-and-driven-animations/animations.html)：文档、局部滚动和视图进度。
3. [styles.css](scripts/26-scroll-snap-and-driven-animations/styles.css)：滚动设置、动画与减少动画规则。

Step 1：在已激活 Python 环境的终端中，从项目根目录进入本技术目录。

```bash
cd content/Web与应用开发/css
```

Step 2：启动本章预览服务。

```bash
python -m http.server 8101 --bind 127.0.0.1
```

Step 3：打开[本章示例首页](http://127.0.0.1:8101/scripts/26-scroll-snap-and-driven-animations/index.html)。

保存修改后刷新页面。

Step 4：在服务终端按 Ctrl+C 停止服务。

## 1 先创建可滚动且可聚焦的区域

滚动吸附（CSS Scroll Snap）让滚动结束的位置偏向指定对齐点。先有尺寸受限、内容产生溢出的滚动容器，吸附设置才有观察意义；scroll-snap-type 本身不会创造溢出。

.snapper 用 Flexbox 横排三张卡片，overflow-x: auto 提供横向滚动。tabindex="0" 让区域可被键盘聚焦，role="region" 与 aria-label 为它命名；每张卡片还保留普通导航链接。

scroll-snap-type: x mandatory 在横轴要求吸附，scroll-snap-align: start 为每张卡片提供起点。不要隐藏滚动条或拦截方向键来模拟本来就有的原生滚动。

```html
<div class="snapper" tabindex="0" role="region" aria-label="三张阅读卡片">
  <section class="slide" id="slide-one"><h2>一：选择主题</h2><p>先确定今天要回答的问题。</p><a href="#slide-two">下一张</a></section>
  <section class="slide" id="slide-two"><h2>二：阅读与尝试</h2><p>阅读材料，再做一个小例子。</p><a href="#slide-three">下一张</a></section>
  <section class="slide" id="slide-three"><h2>三：整理记录</h2><p>记录观察和下一步。</p><a href="#slide-one">返回第一张</a></section>
</div>
```

```css
.snapper {
  display: flex;
  gap: 1rem;
  max-width: 40rem;
  padding: 1rem;
  overflow-x: auto;
  scroll-snap-type: x mandatory;
}
.slide { flex: 0 0 calc(100% - 2rem); min-width: 0; scroll-snap-align: start; }
/* 横向滚动后对齐卡片起点；卡片高度由文字决定，不隐藏滚动条。 */
```

配套文件：[index.html](scripts/26-scroll-snap-and-driven-animations/index.html)、[styles.css](scripts/26-scroll-snap-and-driven-animations/styles.css) · [浏览器预览](http://127.0.0.1:8101/scripts/26-scroll-snap-and-driven-animations/index.html)

## 2 本章使用的属性

| 完整属性名 | 中文名称／含义 | 用途或对象 |
| --- | --- | --- |
| scroll-snap-type | 吸附轴与强度 | 滚动容器 |
| scroll-snap-align | 吸附对齐方式 | 提供吸附位置的子元素 |
| scroll-snap-stop | 吸附越过控制 | 方向性滚动经过的吸附位置 |
| scroll-padding | 滚动可视区内缩 | 调整滚动容器的有效对齐区域 |
| scroll-margin | 吸附区域外扩 | 调整目标元素的吸附范围 |
| overscroll-behavior | 边界滚动行为 | 滚动链与边界效果 |
| scroll-behavior | 滚动行为 | 指定跳转等滚动是否平滑 |
| animation | 动画简写 | 名称、时长、缓动与填充等 |
| animation-timeline | 动画时间线 | 选择驱动动画的进度来源 |
| animation-range | 动画附着范围简写 | 时间线内开始和结束的位置 |
| scroll-timeline-name | 滚动时间线名称 | 声明命名滚动源 |
| scroll-timeline-axis | 滚动时间线轴 | 选择滚动方向 |
| view-timeline-name | 视图时间线名称 | 命名被观察的元素 |
| view-timeline-axis | 视图时间线轴 | 选择观察方向 |
| view-timeline-inset | 视图时间线内缩 | 调整有效观察边界 |

scroll()、view() 是时间线值函数；mandatory、proximity、entry 等是对应语境的关键字。@keyframes 与 @media 是 @ 规则，prefers-reduced-motion 是媒体特征。

## 3 吸附强度、间距与越过限制

x、y 指物理轴，inline、block 随书写模式变化，both 允许双轴。mandatory 要求滚动结束时吸附到有效位置，proximity 只在足够接近时吸附；“足够接近”的距离由浏览器决定，不是固定像素阈值。

scroll-snap-align 可用 start、center、end 或 none；双值时依次表示块方向与行内方向。吸附对齐的是目标的吸附区域（snap area）与容器的吸附视口（snapport）。

scroll-padding 从滚动可视区边缘向内缩小有效区域，scroll-margin 向外扩展目标边框盒的吸附区域。二者不增加真实 padding、margin，也不替容器制造额外滚动距离；末端无法到达的对齐点会受滚动范围限制。

scroll-snap-stop: always 要求带方向的滚动经过相应吸附点时停留，normal 允许越过。它不规定动画时长，也不保证任意一次触控手势都恰好移动一张卡片；直接指定终点的跳转不等同于沿途滚动。

```html
<div class="snapper" tabindex="0" role="region" aria-label="三张阅读卡片">
  <section class="slide" id="slide-one"><h2>一：选择主题</h2><p>先确定今天要回答的问题。</p><a href="#slide-two">下一张</a></section>
  <section class="slide" id="slide-two"><h2>二：阅读与尝试</h2><p>阅读材料，再做一个小例子。</p><a href="#slide-three">下一张</a></section>
  <section class="slide" id="slide-three"><h2>三：整理记录</h2><p>记录观察和下一步。</p><a href="#slide-one">返回第一张</a></section>
</div>
```

```css
.snapper { scroll-padding: 1rem; }
.slide { scroll-margin: 0.25rem; scroll-snap-stop: always; }
/* 中间卡片起点与可视区边缘留有空间；两种scroll间距都不是实际盒模型间距。 */
```

配套文件：[index.html](scripts/26-scroll-snap-and-driven-animations/index.html)、[styles.css](scripts/26-scroll-snap-and-driven-animations/styles.css) · [浏览器预览](http://127.0.0.1:8101/scripts/26-scroll-snap-and-driven-animations/index.html)

## 4 长内容、键盘与滚动链

长文章示例使用 proximity，让读者能在段落中间停留。不能把“吸附元素高于视口”一概说成内容无法读到：规范允许在足够大的吸附区域内部平移；但不相邻吸附点之间的大段内容、不同屏幕尺寸与实现差异仍可能造成可达性问题。

滚动链（scroll chaining）指内层到边界后，滚动继续传给外层。overscroll-behavior: contain 阻断链式传播，通常保留局部回弹；none 还抑制默认边界效果，auto 沿用默认行为。这些值还可能影响下拉刷新或滑动导航，不能无差别施加到全站。

本例让局部区域聚焦后用方向键、PageDown 与 Tab 操作，并将鼠标移出区域再滚动外部文档。滚动链行为应使用真实滚轮或触控检查，脚本直接给 scrollTop 赋值不能证明它被阻断。

```html
<div class="reading-box" tabindex="0" role="region" aria-label="长内容滚动示例">
  <article class="reading-section">
    <h2>一段较长的内容</h2>
    <p>这一节故意超过滚动区域高度，用来检查中间文字仍能读到。</p>
    <p>记录问题，再记录观察；不要让滚动效果把重要内容跳过去。</p>
    <p>试着使用方向键或 PageDown 继续向下阅读。</p>
    <p class="middle-marker">中间标记：这句话应该可以完整滚动到可见区域。</p>
    <p>内容不应因为没有单独的吸附点而变得无法到达。</p>
    <p>最后一段也保留在正常流中。</p>
  </article>
  <article class="reading-section"><h2>下一节</h2><p>在区域边缘继续滚动，比较滚动是否传到外部页面。</p></article>
</div>
```

```css
.reading-box {
  height: 14rem;
  overflow: auto;
  scroll-snap-type: y proximity;
  overscroll-behavior: contain;
}
.reading-section { scroll-snap-align: start; padding: 1rem; }
/* 长内容用proximity：在中间停留阅读；到边缘后观察滚动链是否被阻断。 */
```

配套文件：[index.html](scripts/26-scroll-snap-and-driven-animations/index.html)、[styles.css](scripts/26-scroll-snap-and-driven-animations/styles.css) · [浏览器预览](http://127.0.0.1:8101/scripts/26-scroll-snap-and-driven-animations/index.html)

## 5 滚动进度时间线与文档条带

滚动进度时间线（scroll progress timeline）把某个滚动轴上的当前位置映射为进度：本例从根滚动容器顶部的 0% 到底部的 100%。没有该轴的滚动范围时，时间线可能不活动；空页面不会凭时间自动走完动画。

scroll(root block) 显式选择根滚动容器的块方向。scroll() 也可选择最近祖先滚动容器，或元素自身的滚动容器；选错滚动源会出现“页面在滚、动画不变”。

animation 简写会把之前的 animation-timeline 重置成 auto，恢复默认时间驱动；所以必须先写 animation，再写 animation-timeline。animation-range 同样应放在动画简写之后，避免范围重置。

示例用 auto 时长让动画占满它在进度时间线上的范围；进度由滚动决定，不按经过的秒数自动播放。linear 按进度线性插值，both 在范围两端保留关键帧状态。本例要求支持所写语法的浏览器，不添加针对旧实现的时长保护。

条带只表示滚动位置，不证明读者已经阅读多少文字，因此使用 aria-hidden="true"，不把它冒充有精确语义的阅读完成率控件。

```html
<div class="page-meter" aria-hidden="true"></div>
```

```css
@keyframes fill-progress { from { transform: scaleX(0); } to { transform: scaleX(1); } }
.page-meter {
  display: block;
  position: fixed;
  inset: 0 0 auto;
  height: 0.4rem;
  background: #0645ad;
  transform-origin: left;
  z-index: 1;
  animation: fill-progress auto linear both;
  animation-timeline: scroll(root block);
}
/* 未要求减少运动时：页面顶部 scaleX(0)，底部 scaleX(1)。
   往回滚动时条带缩短，停住滚动后不会随经过的秒数继续增长。 */
```

配套文件：[animations.html](scripts/26-scroll-snap-and-driven-animations/animations.html)、[styles.css](scripts/26-scroll-snap-and-driven-animations/styles.css) · [浏览器预览](http://127.0.0.1:8101/scripts/26-scroll-snap-and-driven-animations/animations.html)

## 6 命名局部滚动时间线

scroll-timeline-name 在实际滚动容器上命名 --local，scroll-timeline-axis 指定 block。内部条带通过 animation-timeline: --local 关联这个滚动源；名字是时间线标识，不是 CSS 自定义属性。

条带位于命名容器内部，并用 sticky 留在滚动区域顶部。这样可以分别观察外部页面和内层区域，避免隐式选择最近容器时弄错对象。

命名时间线有可见范围，不能假设任意远处兄弟元素都能解析这个名字；跨子树共享涉及额外的 timeline-scope 等机制，本例不需要。本例直接使用命名时间线，要求浏览器支持这一功能。

```html
<div class="local-reader" tabindex="0" role="region" aria-label="局部阅读进度">
  <div class="local-meter" aria-hidden="true"></div>
  <p>第 1 段局部阅读内容：只有这个边框内的滚动决定它自己的进度条。</p>
<p>第 2 段局部阅读内容：只有这个边框内的滚动决定它自己的进度条。</p>
<p>第 3 段局部阅读内容：只有这个边框内的滚动决定它自己的进度条。</p>
<p>第 4 段局部阅读内容：只有这个边框内的滚动决定它自己的进度条。</p>
<p>第 5 段局部阅读内容：只有这个边框内的滚动决定它自己的进度条。</p>
<p>第 6 段局部阅读内容：只有这个边框内的滚动决定它自己的进度条。</p>
<p>第 7 段局部阅读内容：只有这个边框内的滚动决定它自己的进度条。</p>
<p>第 8 段局部阅读内容：只有这个边框内的滚动决定它自己的进度条。</p>
<p>第 9 段局部阅读内容：只有这个边框内的滚动决定它自己的进度条。</p>
<p>第 10 段局部阅读内容：只有这个边框内的滚动决定它自己的进度条。</p>
<p>第 11 段局部阅读内容：只有这个边框内的滚动决定它自己的进度条。</p>
<p>第 12 段局部阅读内容：只有这个边框内的滚动决定它自己的进度条。</p>
</div>
```

```css
.local-reader { height: 14rem; overflow: auto; }
.local-reader { scroll-timeline-name: --local; scroll-timeline-axis: block; }
.local-meter {
  display: block;
  position: sticky;
  top: 0;
  height: 0.4rem;
  background: #0b715f;
  transform-origin: left;
  animation: fill-progress auto linear both;
  animation-timeline: --local;
}
/* 进度条位于命名滚动容器内部；只滚外部文档不应推进局部进度。 */
```

配套文件：[animations.html](scripts/26-scroll-snap-and-driven-animations/animations.html)、[styles.css](scripts/26-scroll-snap-and-driven-animations/styles.css) · [浏览器预览](http://127.0.0.1:8101/scripts/26-scroll-snap-and-driven-animations/animations.html)

## 7 视图进度与动画范围

视图进度时间线（view progress timeline）关注目标元素在最近祖先滚动容器中的相对位置。它跟踪几何相交过程，不测量用户是否看见，也不判断被其他元素遮挡后的可见像素。

view(block 0px) 以当前动画元素为被观察对象，按块方向观察，显式使用 0px 内缩。view() 没有选择任意根容器的参数，使用最近祖先滚动容器；若省略内缩，auto 会关联相应 scroll-padding。

先固定本例“色块比滚动区域矮、内缩为 0”的条件，再读 entry 的两个端点。整个视图时间线覆盖进入到离开的过程，animation-range: entry 0% entry 100% 只取进入这一段：开始相交时启动，恰好完全进入时完成。

cover 表示覆盖整个穿越过程，contain 表示较小者完全处于较大者内部的阶段，entry 与 exit 分别表示进入和离开阶段。百分比是各自命名范围内的位置，不能把 entry 100% 理解成已经滚完整篇文档；目标比视口大时也不能仍按“整个目标完全可见”来解释。

本例直接声明时间线和范围，文字始终正常显示，只对装饰色块改变透明度。返回上方再滚入，会沿相同进度反向变化，不是只触发一次的进入事件。

![同一较矮色块相对滚动区域的三个位置：entry0刚接触下边界、entry50进入一半、entry100恰好完全进入。](image/illustration/26-01-view-entry-progress.svg)

图 1：依据篇末 Scroll-driven Animations 的 entry 范围定义自绘，本节色块的进入阶段。框表示滚动区域，虚线标下边界，箭头表示向下滚动时目标在视口中的移动方向；不是时间均分或整篇阅读百分比。

缓慢滚动下方示例，找到透明度刚开始变化与完全不透明的位置；再反向滚动，检查变化是否沿同一几何进度返回。

```html
<section class="story-section">
  <h2>视图进度观察</h2>
  <p>文字始终可读，下面的色块只承担视觉效果。</p>
  <div class="reveal" aria-hidden="true"></div>
</section>
```

```css
.reveal { height: 7rem; background: #0b715f; opacity: 1; }
@keyframes reveal-tone { from { opacity: 0.25; } to { opacity: 1; } }
.reveal {
  animation: reveal-tone auto linear both;
  animation-timeline: view(block 0px);
  animation-range: entry 0% entry 100%;
}
/* 色块进入视口的过程中变为完全不透明，完全进入后保持；正文没有透明度动画。 */
```

配套文件：[animations.html](scripts/26-scroll-snap-and-driven-animations/animations.html)、[styles.css](scripts/26-scroll-snap-and-driven-animations/styles.css) · [浏览器预览](http://127.0.0.1:8101/scripts/26-scroll-snap-and-driven-animations/animations.html)

## 8 命名视图时间线区分观察对象

view-timeline-name 在 .view-subject 上命名 --chapter；内部 .named-accent 使用这个名字。观察对象是整节，动画对象是装饰线，二者不必相同。

view-timeline-inset: 0 明确使用滚动可视区边界；正内缩将有效边界向内收，负值向外扩。它改变进度的起止几何关系，不替元素添加实际 padding。

本例与前节都只使用进入阶段，但一个按色块进入计算，一个按整个章节进入计算。不要因为两条规则都写 entry 就期待它们在相同滚动位置完成。

```html
<section class="view-subject story-section">
  <h2>以章节作为观察对象</h2>
  <div class="named-accent" aria-hidden="true"></div>
  <p>整节进入视口的进度用于改变这条装饰线。</p>
</section>
```

```css
.named-accent { height: 0.5rem; background: #0645ad; transform-origin: left; }
.view-subject { view-timeline-name: --chapter; view-timeline-axis: block; view-timeline-inset: 0; }
.named-accent {
  animation: fill-progress auto linear both;
  animation-timeline: --chapter;
  animation-range: entry 0% entry 100%;
}
/* 未要求减少运动时：section 进入视口期间，内部装饰线从零伸展到全宽。
   被观察的是 section，发生变换的是内部装饰线。 */
```

配套文件：[animations.html](scripts/26-scroll-snap-and-driven-animations/animations.html)、[styles.css](scripts/26-scroll-snap-and-driven-animations/styles.css) · [浏览器预览](http://127.0.0.1:8101/scripts/26-scroll-snap-and-driven-animations/animations.html)

## 9 减少动画与状态检查

prefers-reduced-motion: reduce 表示用户请求减少非必要运动。本例关闭所有滚动装饰动画，并显式恢复完整透明度和无变换；同时把卡片吸附改为 proximity，这是本例降低强制移动的选择，不是该媒体特征自动完成的行为。

不能只把 animation-duration 改成 0s 就认为滚动驱动动画关闭了，也不能关动画后把元素留在基础 opacity: 0。本例基础内容始终可读，装饰进度条在普通偏好下显示、reduce时隐藏。

在浏览器中分别观察普通与 reduce 模式：根滚动到顶部、中段、底部；局部滚动到一半；色块从未进入到完全进入。核对条带缩放与色块透明度，而不只看 CSS.supports 返回值。

临时禁用动画规则后，页面必须仍能滚动、所有正文可读，链接与焦点可操作。滚动驱动并不自动保证低成本或所有浏览器相同；本章没有给出未经测量的帧率结论。

```html
<div class="page-meter" aria-hidden="true"></div>
```

```css
@media (prefers-reduced-motion: reduce) {
  .page-meter, .local-meter { display: none; animation: none; }
  .reveal, .named-accent { animation: none; opacity: 1; transform: none; }
  .snapper { scroll-snap-type: x proximity; scroll-behavior: auto; }
  .slide { scroll-snap-stop: normal; }
}
/* 切换减少动画偏好并重新滚动：装饰保持静态，全文和卡片链接仍可读、可操作。 */
```

配套文件：[animations.html](scripts/26-scroll-snap-and-driven-animations/animations.html)、[styles.css](scripts/26-scroll-snap-and-driven-animations/styles.css) · [浏览器预览](http://127.0.0.1:8101/scripts/26-scroll-snap-and-driven-animations/animations.html)

## 本章小结

- 吸附需要真实滚动范围；容器决定轴与强度，目标决定对齐和越过限制。
- scroll-padding 与 scroll-margin 改变吸附几何，不改变盒模型间距。
- 滚动时间线取滚动位置，视图时间线取元素相对滚动可视区的位置。
- animation 简写之后再设置时间线与范围，减少动画模式显式恢复静态可读状态。
- 键盘、长内容、滚动链与动画偏好各有不同检查方法。

## 练习

（1）将中间卡片改成 center 对齐，再分别增加 scroll-padding 和 scroll-margin。标准：指出哪个设置改变容器区域、哪个改变目标区域，并检查首尾滚动边界限制。

（2）给长内容增加多个段落，分别用 proximity 与 mandatory 观察中间标记。标准：中间文字可到达，键盘能离开滚动区域；不能把规范的大区域规则与实际浏览器行为混为一谈。

（3）把文档条带的 animation-timeline 移到 animation 之前。标准：在计算样式中找到时间线被重置，修复次序后确认条带再次随滚动位置变化。

（4）切换 reduce 偏好，滚动根文档与局部阅读框，再恢复普通偏好。标准：reduce 时装饰静止、内容完整，普通模式下两种进度来源各自生效。

### 提示

每次修改后保存刷新；测试时分清根滚动与局部滚动。不要用脚本滚动模拟触控惯性结论，也不要把截图生成当成动画全过程已验证。

### 解析

（1）scroll-padding改变容器snapport，scroll-margin改变目标snap area；只加对称间距时center的中心可能不变，因此不能预设每次都产生位移。改为不对称间距再比较，同时保持卡片真实padding、margin和宽度不变。

（2）本例整段article就是吸附区域，区域大于视口时规范允许在内部平移，mandatory并不必然跳过中间文字；仍要用实际方向键、滚轮和Tab核对可达性。

（3）后写animation会把timeline重置为auto，失去滚动关联；当前示例时长也为auto，默认时间线下相当于零时长，通常直接呈现终点。把timeline恢复到简写后，顶/中/底应重新对应0/约0.5/1。

（4）reduce下两个meter隐藏，reveal透明度1、named-accent无变换；恢复普通偏好后只滚内层应只推进局部条带，滚外层应推进文档条带。


## 参考与引用来源

| 来源站点 | 核查定位与对应内容 |
| --- | --- |
| W3C | [CSS Scroll Snap Level 1 §3–5](https://www.w3.org/TR/css-scroll-snap-1/#overview) 的吸附区域、轴和间距；[§5.2.2](https://www.w3.org/TR/css-scroll-snap-1/#snap-overflow) 的大吸附区域内滚动；[§5.3](https://www.w3.org/TR/css-scroll-snap-1/#scroll-snap-stop) 的方向性滚动与越过限制；[Scroll-driven Animations Level 1 §2–3](https://www.w3.org/TR/scroll-animations-1/#scroll-timelines) 的滚动／视图进度，[§3.1](https://www.w3.org/TR/scroll-animations-1/#view-timelines-ranges) 的 entry、exit、cover、contain，以及 [附录 A](https://www.w3.org/TR/scroll-animations-1/#animation-range) 的动画附着范围；[CSS Animations 2 §4.1](https://www.w3.org/TR/css-animations-2/#animation-duration) 与 [Scroll-driven Animations §4.1](https://www.w3.org/TR/scroll-animations-1/#finite-timelines) 的auto时长与有限时间线计算。 |
| MDN | [Basic concepts of scroll snap](https://developer.mozilla.org/en-US/docs/Web/CSS/Guides/Scroll_snap/Basic_concepts) 的基础用法；[scroll-padding](https://developer.mozilla.org/en-US/docs/Web/CSS/Reference/Properties/scroll-padding)、[scroll-margin](https://developer.mozilla.org/en-US/docs/Web/CSS/Reference/Properties/scroll-margin)、[scroll-snap-stop](https://developer.mozilla.org/en-US/docs/Web/CSS/Reference/Properties/scroll-snap-stop) 的配置对象；[overscroll-behavior](https://developer.mozilla.org/en-US/docs/Web/CSS/Reference/Properties/overscroll-behavior#values) 的滚动链和边界效果；[Scroll-driven animation timelines](https://developer.mozilla.org/en-US/docs/Web/CSS/Guides/Scroll-driven_animations/Timelines) 的命名／匿名滚动源、视图观察对象与内缩；[animation-timeline 的 Description](https://developer.mozilla.org/en-US/docs/Web/CSS/Reference/Properties/animation-timeline#description) 的简写重置；[animation-duration](https://developer.mozilla.org/en-US/docs/Web/CSS/Reference/Properties/animation-duration#values) 的 auto 与实现条件；[animation-range](https://developer.mozilla.org/en-US/docs/Web/CSS/Reference/Properties/animation-range) 的范围语法；[scroll-timeline-name](https://developer.mozilla.org/en-US/docs/Web/CSS/Reference/Properties/scroll-timeline-name)、[view-timeline-name](https://developer.mozilla.org/en-US/docs/Web/CSS/Reference/Properties/view-timeline-name) 的命名关联；[prefers-reduced-motion](https://developer.mozilla.org/en-US/docs/Web/CSS/Reference/At-rules/@media/prefers-reduced-motion) 的用户偏好。 |
| Python 3.12 | [http.server 命令行](https://docs.python.org/3.12/library/http.server.html#command-line-interface) 的本地服务。 |
